In [ ]:
import cv2
import numpy as np
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import RedirectResponse
from ultralytics import YOLO
import uvicorn
import threading

app = FastAPI(title="Smart Object Classifier API")

@app.get("/")
def root():
    return {
        "status": "working", 
        "message": "API запущен."
    }

DETECTOR_PATH = 'yolov8n.pt' 
CLASSIFIER_PATH = 'runs/classify/my_animal_model2/weights/best.pt'

print("Загрузка моделей...")
detector = YOLO(DETECTOR_PATH)
classifier = YOLO(CLASSIFIER_PATH)

def process_image_pipeline(frame: np.ndarray) -> dict:
    det_results = detector.predict(frame, verbose=False, conf=0.4)
    best_box = None
    max_area = 0

    if len(det_results[0].boxes) > 0:
        for box in det_results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            area = (x2 - x1) * (y2 - y1)
            if area > max_area:
                max_area = area
                best_box = (x1, y1, x2, y2)

    if best_box:
        x1, y1, x2, y2 = best_box
        h, w, _ = frame.shape
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        object_crop = frame[y1:y2, x1:x2]

        if object_crop.size > 0:
            cls_results = classifier.predict(object_crop, verbose=False)
            probs = cls_results[0].probs
            top1_idx = probs.top1
            conf = probs.top1conf.item()
            class_name = cls_results[0].names[top1_idx]

            return {
                "status": "success",
                "object_found": True,
                "box": [int(x1), int(y1), int(x2), int(y2)],
                "class_name": class_name,
                "confidence": round(conf, 4)
            }

    return {
        "status": "success",
        "object_found": False,
        "message": "Searching for animal..."
    }


@app.post("/predict")
def predict(file: UploadFile = File(...)):
    contents = file.file.read()
    if not contents:
        return {"status": "error", "message": "Получен пустой файл."}
        
    nparr = np.frombuffer(contents, np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    if frame is None:
        return {"status": "error", "message": "Не удалось декодировать изображение."}
    
    return process_image_pipeline(frame)


def run_fastapi_server():
    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()

if __name__ == "__main__":
    server_thread = threading.Thread(target=run_fastapi_server, daemon=True)
    server_thread.start()
    print("Сервер успешно запущен в фоновом потоке!")
    print("Ссылка на статус: http://127.0.0.1:8000")
    print("Ссылка на документацию: http://127.0.0.1:8000/docs")

Загрузка моделей...
Сервер успешно запущен в фоновом потоке!
Ссылка на статус: http://127.0.0.1:8000
Ссылка на документацию: http://127.0.0.1:8000/docs


INFO:     Started server process [38988]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8000): обычно разрешается только одно использование адреса сокета (протокол/сетевой адрес/порт)
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:64544 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64546 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64547 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64548 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64549 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64550 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64551 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64552 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64553 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64554 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64555 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64556 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64557 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64558 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64559 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64560 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:64561 - "POST /predi